##  Setup & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


In [ ]:
DATA_DIR   = Path("/content/drive/MyDrive/TTC_Dataset/raw")
EXPORT_DIR = Path("/content/drive/MyDrive/TTC_Dataset/output")
EXPORT_DIR.mkdir(exist_ok=True)
END_DATE   = "2026-06-30"

## Handling & Combining Data

#Merging Dataset
2022-2024 data are xlsx files while 2025and 2026 till June is a csv file.

Loaded the source files and merged as one Dataframe.


In [ ]:
# notice that the 2022-2024 data are xlsx files while 2025+ is a csv file
xlsx_sources = [
    (DATA_DIR/"ttc-subway-delay-data-2022.xlsx", "2022"),
    (DATA_DIR/"ttc-subway-delay-data-2023.xlsx", "2023"),
    (DATA_DIR/"ttc-subway-delay-data-2024.xlsx", "Subway"),
]
dfs = [pd.read_excel(p, sheet_name=s) for p, s in xlsx_sources]

# 2025+ csv has an extra _id column so we must drop it
df_2025 = pd.read_csv(DATA_DIR / "TTC Subway Delay Data since 2025.csv")
df_2025 = df_2025.drop(columns=["_id"], errors="ignore")
dfs.append(df_2025)

df = pd.concat(dfs, ignore_index=True)
print(f"Combined shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range : {pd.to_datetime(df['Date']).min().date()} to {pd.to_datetime(df['Date']).max().date()}")
df.head(10)
master_df=df.copy()

Combined shape : 109,967 rows x 10 columns
Date range : 2022-01-01 to 2026-06-30


# Data Profiling
Data profiling is the process of examining a dataset to understand its structure, completeness, consistency, uniqueness, and overall data quality before performing analysis.

What do you check during data profiling?


*  **Structure**: Number of rows and columns, data types
*   **Missing values**: How many NaN/null values exist?
*  **Duplicates**: Are there duplicate records?
*  **Unique values**: What categories or values exist?
*  **Distribution**: Min, max, mean, median, etc.
*   **Inconsistencies**: For example, YU/BD, YU / BD, and BD/YU representing the same thing
*  **Outliers**: Unusually high or low values
*   **Data quality**: Invalid, inconsistent, or unexpected values












In [ ]:
info_df = pd.DataFrame({
    "dtype"     : df.dtypes,
    "non_null"  : df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_%"    : (df.isna().mean() * 100).round(1),
})
print(info_df.to_string())
print("Min Delay Statistics:")
print(df["Min Delay"].describe().round(2).to_string())
print(df["Min Gap"].describe().round(2).to_string())

            dtype  non_null  null_count  null_%
Date       object    109967           0     0.0
Time       object    109967           0     0.0
Day        object    109967           0     0.0
Station    object    109967           0     0.0
Code       object    109967           0     0.0
Min Delay   int64    109967           0     0.0
Min Gap     int64    109967           0     0.0
Bound      object     72196       37771    34.3
Line       object    109693         274     0.2
Vehicle     int64    109967           0     0.0
Min Delay Statistics:
count    109967.00
mean          2.99
std          11.59
min           0.00
25%           0.00
50%           0.00
75%           4.00
max         900.00
count    109967.00
mean          4.28
std          11.54
min           0.00
25%           0.00
50%           0.00
75%           8.00
max         906.00


In [ ]:
df.isna().sum()

,0
Date,0
Time,0
Day,0
Station,0
Code,0
Min Delay,0
Min Gap,0
Bound,37771
Line,274
Vehicle,0


## Data Cleaning
- Notice that there are multiple variants of the same Line in the 'Line' column (eg. YU, YUS, Line 1m etc.)
- Line 3 (SRT) is discontinued, so it must be dropped
- Many typos in the source data, so it needs replacement with the right spelling
- Some 'Line' data is missing but can be retrieved by mapping


In [ ]:
# Bound column has more null rows and duplicate information so dropping it.
df.drop(columns=["Bound"], inplace=True)
df.head()


,Date,Time,Day,Station,Code,Min Delay,Min Gap,Line,Vehicle
0,2022-01-01 00:00:00,15:59,Saturday,LAWRENCE EAST STATION,SRDP,0,0,SRT,3023
1,2022-01-01 00:00:00,02:23,Saturday,SPADINA BD STATION,MUIS,0,0,BD,0
2,2022-01-01 00:00:00,22:00,Saturday,KENNEDY SRT STATION TO,MRO,0,0,SRT,0
3,2022-01-01 00:00:00,02:28,Saturday,VAUGHAN MC STATION,MUIS,0,0,YU,0
4,2022-01-01 00:00:00,02:34,Saturday,EGLINTON STATION,MUATC,0,0,YU,5981


In [ ]:
df["Line"].value_counts()

,count
Line,
YU,57074
BD,44926
SHP,4054
SRT,1935
YU/BD,1463
YUS/BD,60
YU / BD,30
YUS,24
YU/ BD,23


Line 3 / SRT → train service ended July 24, 2023. So, we no longer needed the corresponding data for analysis.
Dropping the Line 3/SRT rows from dataframe.

In [ ]:
# Line 3  was discontinued, so corresponding rows are dropped.
length_before_drop = len(df)
df = df[df["Line"] != "SRT"].copy()
print(f"Line 3 service  Rows dropped: {length_before_drop - len(df)}")

Line 3 service  Rows dropped: 1935


Multiple variants of the same line are causing inconsistency in data.

---

Normalizing the Line column.

In [ ]:
# normalize line column since there are many unecessary varaints of the same line
LINE_NORM = {
    # YU (line 1) variants
    "YUS": "YU", "LINE 1": "YU",
    # BD (line 2) variants
    "B/D": "BD", "BD LINE 2": "BD", "BLOOR DANFORTH": "BD", "LINE 2 SHUTTLE": "BD", "LINE 2 - BLOOR DANFORT" :"BD",
    # SHP (line 4) variants
    "SHEP": "SHP",
    # Multi-Line variants
    "YUS/BD": "YU/BD",  "YU / BD": "YU/BD",  "YU/ BD": "YU/BD",
    "BD/YU":  "YU/BD",  "YU & BD": "YU/BD",  "YU/BD LINES": "YU/BD",
    "BD/YUS": "YU/BD",  "YU/BD LINE": "YU/BD", "YUS / BD": "YU/BD",
    "Y/BD":   "YU/BD",  "YUS AND BD": "YU/BD", "YUS/ BD": "YU/BD",
    "BD/ YU": "YU/BD",  "BD / YU": "YU/BD",   "BD/ YUS": "YU/BD",
    "YU -BD LINES": "YU/BD", "BLOOR DANFORTH & YONGE": "YU/BD",
    "ONGE-UNIVERSITY AND BL": "YU/BD",
    "YUS/ BD/ SHP": "YU/BD/SHP","YUS/BD/SHP": "YU/BD/SHP", "YU/BD/SHP": "YU/BD/SHP",
}
df["Line"] = df["Line"].replace(LINE_NORM)

VALID_LINES = {"YU", "BD", "SHP", "YU/BD", "YU/BD/SHP"}

length_before_drop = len(df)
df = df[df["Line"].isin(VALID_LINES)].copy()
print(f"Rows with invalid 'Line' values are dropped {length_before_drop - len(df):,} ")
print(df["Line"].value_counts().to_string())

#df.to_csv(EXPORT_DIR / "TTC_Subway_Delay_Data_2022_2026.csv")

Rows with invalid 'Line' values are dropped 298 
Line
YU           57100
BD           44932
SHP           4055
YU/BD         1636
YU/BD/SHP       11


Found some invalid names in Station column. Removed them from dataset.




In [ ]:
# remove invalid stations
INVALID_PATTERNS = [
    r"HILLCREST", r"CARHOUSE", r"\bYARD\b", r"SHOPS\b",
    r"BUILDING\b", r"COMPLEX\b", r"\bGATE\b", r"OFFICES\b",
    r"\bOPS\b", r"INGLIS", r"GO PROTOCOL", r"MCBRIEN",
    r"GUNN\b", r"SUBWAY CLOSURE", r"TRACK LEVEL ACTIVITY",
]
pattern = "|".join(INVALID_PATTERNS)
mask = df["Station"].str.upper().str.contains(pattern, regex=True, na=False)
print(f"Rows with valid station names: {mask.value_counts()}")
length_before_drop = len(df)
df = df[~mask].copy()

Rows with valid station names: Station
False    106930
True        804
Name: count, dtype: int64


### Fill Missing Line Values

- Count missing values in the `Line` column.
- Use the station-to-line mapping to fill missing `Line` values.
- Preserve existing non-null `Line` values.
- Remove rows where the `Line` value remains unresolved.
- Create a clean dataset for further analysis and visualization.

In [ ]:
#station_line_map is dictionary-like Station to Line lookup,will use it to fill missing Line values
station_line_map = (
    df.dropna(subset=["Line"]) #Removes rows where the Line column is missing (NaN)
    .groupby("Station")["Line"] #Groups all rows belonging to the same station.After grouping by station, we're specifically interested in the Line values within each station.
    .agg(lambda x: x.mode().iloc[0]) #For each station, find the mode — the most frequently occurring Line. iloc[0] takes the first result.
)
#print(station_line_map)
missing_before = df["Line"].isna().sum()
# fill remaining null line values based on mapping
if missing_before:
    df["Line"] = df["Line"].fillna(df["Station"].map(station_line_map)) #Existing Line values are kept, and only missing Line values are replaced using the station mapping.
    missing_after = df["Line"].isna().sum()
    filled = missing_before - missing_after
    print(f"Filled {filled} missing Line values from station mapping")
    print(f" Still unresolved : {missing_after}")
    df = df.dropna(subset=["Line"]).copy() # drop any that couldn't be resolved

print(f"missing_before: {missing_before}")
print(f"Shape after cleaning : {df.shape[0]:,} rows")

missing_before: 0
Shape after cleaning : 106,930 rows


In [ ]:
# Have typos in some codes, so corrected here
TYPO_MAP = {
    "MUNCA": "MUNOA",
    "TUNCA": "TUNOA",
    "TRNCA": "TRNOA",
}
df["Code"] = df["Code"].replace(TYPO_MAP)

# merge code Code_Descriptions
df_codes_raw = pd.read_excel(
    DATA_DIR / "ttc-subway-delay-codes.xlsx",
    sheet_name="Sheet 1", header=None
)

# the xlsx has an empty row at row index 1; data starts at row 2
# subway codes are in columns 2 (code) and 3 (description)
df_codes = df_codes_raw[[2, 3]].iloc[2:].copy()
df_codes.columns = ["Code", "Code_Description"]
df_codes = df_codes.dropna(subset=["Code"]).reset_index(drop=True)
df_codes["Code"] = df_codes["Code"].astype(str).str.strip()
df_codes["Code_Description"] = df_codes["Code_Description"].astype(str).str.strip()
df_codes.columns

Index(['Code', 'Code_Description'], dtype='object')

In [ ]:
def categorize_code(description):

    description = str(description).lower()

    # 1. Passenger & Medical
    if any(keyword in description for keyword in [
        "injured or ill customer",
        "passenger related",
        "passenger",
        "disorderly patron",
        "passenger assistance"
    ]):
        return "Passenger & Medical"

    # 2. Safety & Security
    elif any(keyword in description for keyword in [
        "assault",
        "robbery",
        "bomb threat",
        "emergency alarm",
        "suspicious package",
        "held by police",
        "unauthorized at track",
        "fire",
        "smoke",
        "train in contact with person"
    ]):
        return "Safety & Security"

    # 3. Train & Equipment
    elif any(keyword in description for keyword in [
        "air conditioning",
        "alternating current",
        "atc rc&s equipment",
        "brakes",
        "body",
        "compressed air",
        "chopper control",
        "couplers",
        "door problems - faulty equipment",
        "high voltage",
        "lighting system",
        "low voltage",
        "propulsion",
        "speed control equipment",
        "trainline",
        "traction motors",
        "trucks",
        "cab doors",
        "warning alarm systems",
        "work vehicle",
        "equipment",
        "train controls",
        "doors open in error"
    ]):
        return "Train & Equipment"

    # 4. Infrastructure & Signals
    elif any(keyword in description for keyword in [
        "signal",
        "signalling",
        "track",
        "traction power",
        "rail related",
        "structure",
        "track circuit",
        "track switch",
        "station",
        "escalator",
        "elevator",
        "scada",
        "radio system",
        "data communications",
        "beacon",
        "work zone"
    ]):
        return "Infrastructure & Signals"

    # 5. Operations & Workforce
    elif any(keyword in description for keyword in [
        "operator",
        "employee",
        "labour",
        "work refusal",
        "training department",
        "transit control",
        "supervisory",
        "crew",
        "transportation department",
        "maintenance error"
    ]):
        return "Operations & Workforce"

    # 6. Environmental & Cleanliness
    elif any(keyword in description for keyword in [
        "weather",
        "storm",
        "unsanitary",
        "debris",
        "force majeure",
        "miscellaneous",
        "other",
        "consequential"
    ]):
        return "Environmental & Cleanliness"

    else:
        return "Other"

df_codes["Code_Category"] = df_codes["Code_Description"].apply(categorize_code)
df_codes['Code_Category'].value_counts()

,count
Code_Category,
Infrastructure & Signals,31
Train & Equipment,24
Other,22
Operations & Workforce,19
Safety & Security,13
Environmental & Cleanliness,12
Passenger & Medical,8


In [ ]:
df = df.merge(df_codes, on="Code", how="left")
df["Code_Description"] = df["Code_Description"].fillna("Unknown")

matched_pct = (df["Code_Description"] != "Unknown").mean()
print(f"Codes matched to Code_Description: {matched_pct:.1%}")

Codes matched to Code_Description: 98.6%


In [ ]:
# assigning description categories for future visualization
def assign_category(desc):
    d = str(desc).lower()
    #print(d)
    if any(w in d for w in ["assault","robbery"]):
        return "Assault / Robbery"
    if any(w in d for w in ["disorderly"]):
        return "Disorderly Patron"
    if any(w in d for w in ["injured","ill"]):
        return "Medical Emergency"
    if any(w in d for w in ["passenger other"]):
        return "Passenger Related"
    if any(w in d for w in ["alarm" ,"activated","activation"]):
        return "Alarm Activated"
    if any(w in d for w in ["door","doors","faulty"]):
        return "Train Door issue"
    if any(w in d for w in ["miscellaneous"]):
        return "Miscellaneous"
    if any(w in d for w in ["operator"]):
        return "Operator Related"
    if any(w in d for w in ["weather", "ice", "snow", "flood"]):
        return "Weather / Environmental"
    if any(w in d for w in ['signal', 'signalling', 'signaling',"signals"]):
        return 'Signals / Signalling'
    if any(w in d for w in ['track', 'rail', 'switch', 'structure']):
        return 'Track / Infrastructure'
    if any(w in d for w in ["station",'escalator/elevator']):
        return 'Station / Accessibility'
    if any(w in d for w in ["debris", "unauthorized", "trespass", "police", "polce","security","fire"]):
        return "Security / Safety"
    if any(w in d for w in ["switch", "atc ", "power failure", "electrical","voltage","high","low","radio"]):
        return "Power Related"
    if any(w in d for w in ["brake", "propulsion", "motor", "coupler","equipment"]):
        return 'Mechanical / Train Equipment'
    if any(w in d for w in ["employee", "department"]):
        return 'Department / Employee Related'
    if any(w in d for w in ["unsanitary"]):
        return 'Unsanitary Vehicle'
    return 'Other / Unknown'

df["Code_Category"] = df["Code_Description"].apply(assign_category)


For Visualization and understand, grouped the code descriptions into logical categories an added as a column `Code_Category`

In [ ]:
df['Code_Category'].value_counts()

,count
Code_Category,
Passenger & Medical,50940
Operations & Workforce,12188
Environmental & Cleanliness,11178
Infrastructure & Signals,9728
Other,9004
Safety & Security,8121
Train & Equipment,4313


## Datetime & Parsing Features

In [ ]:
# Date arrives as datetime64 from xlsx and as string from csv, so we normalise
df["Date_str"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")
df["Time_str"] = df["Time"].astype(str).str.strip().str[:5]

df["Datetime"] = pd.to_datetime(
    df["Date_str"] + " " + df["Time_str"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)
to_drop = df["Datetime"].isna().sum()
print(f"Null Datetime rows: {to_drop}")
if to_drop:
    print(f"Dropping {to_drop} rows with unparseable datetime")
    df = df.dropna(subset=["Datetime"]).copy()


Null Datetime rows: 0


### Creating Date and Time Features

The `Datetime` column is used to extract and create additional date and time features for analyzing TTC delay patterns.

The following features are created:
- **Year** – Calendar year of the delay.
- **Month** – Numeric month of the delay.
- **Day_Num** – Day of the month.
- **Hour** – Hour of the day, used for time-based analysis.
- **Weekday** – Numeric day of the week, where Monday = 0 and Sunday = 6.
- **DayName** – Name of the day of the week.
- **Peak_Hour** – Binary indicator identifying TTC peak periods: 7:00–10:00 and 16:00–19:00.
- **Weekend** – Binary indicator where 1 represents Saturday/Sunday and 0 represents weekdays.
- **Season** – Categorizes each record into Winter, Spring, Summer, or Fall based on the month.

These features support analysis of TTC delays by **year, month, day, hour, weekday/weekend, peak periods, and season**, and can be used for both exploratory analysis and Power BI visualizations.





In [ ]:
# create and extract time-based features for modelling
df["Year"]      = df["Datetime"].dt.year
df["Month"]     = df["Datetime"].dt.month
df["Day_Num"]   = df["Datetime"].dt.day
df["Hour"]      = df["Datetime"].dt.hour
df["Weekday"]   = df["Datetime"].dt.weekday # 0 = Monday, 6 = Sunday
df["DayName"]   = df["Datetime"].dt.day_name()
df["Peak_Hour"] = df["Hour"].apply(lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 19) else 0)
df["Weekend"]   = (df["Weekday"] >= 5).astype(int)
df["Season"]    = df["Month"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring", 5:"Spring",
    6:"Summer",  7:"Summer", 8:"Summer",
    9:"Fall",   10:"Fall",  11:"Fall",
})

# drop helper columns
df = df.drop(columns=["Date_str", "Time_str"])

print(f"Final shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range  : {df['Datetime'].min().date()} to {df['Datetime'].max().date()}")


Final shape : 106,930 rows x 21 columns
Date range  : 2022-01-01 to 2026-06-30


In [ ]:
df.head()

,Date,Time,Day,Station,Code,Min Delay,Min Gap,Line,Vehicle,Code_Description,Code_Category,Datetime,Year,Month,Day_Num,Hour,Weekday,DayName,Peak_Hour,Weekend,Season
0,2022-01-01 00:00:00,02:23,Saturday,SPADINA BD STATION,MUIS,0,0,BD,0,Injured or ill Customer (In Station) - Transpo...,Passenger & Medical,2022-01-01 02:23:00,2022,1,1,2,5,Saturday,0,1,Winter
1,2022-01-01 00:00:00,02:28,Saturday,VAUGHAN MC STATION,MUIS,0,0,YU,0,Injured or ill Customer (In Station) - Transpo...,Passenger & Medical,2022-01-01 02:28:00,2022,1,1,2,5,Saturday,0,1,Winter
2,2022-01-01 00:00:00,02:34,Saturday,EGLINTON STATION,MUATC,0,0,YU,5981,ATC Project,Other,2022-01-01 02:34:00,2022,1,1,2,5,Saturday,0,1,Winter
3,2022-01-01 00:00:00,05:40,Saturday,QUEEN STATION,MUNOA,0,0,YU,0,No Operator Immediately Available - Not E.S.A....,Operations & Workforce,2022-01-01 05:40:00,2022,1,1,5,5,Saturday,0,1,Winter
4,2022-01-01 00:00:00,06:56,Saturday,DAVISVILLE STATION,MUNOA,0,0,YU,0,No Operator Immediately Available - Not E.S.A....,Operations & Workforce,2022-01-01 06:56:00,2022,1,1,6,5,Saturday,0,1,Winter


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106930 entries, 0 to 106929
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date              106930 non-null  object        
 1   Time              106930 non-null  object        
 2   Day               106930 non-null  object        
 3   Station           106930 non-null  object        
 4   Code              106930 non-null  object        
 5   Min Delay         106930 non-null  int64         
 6   Min Gap           106930 non-null  int64         
 7   Line              106930 non-null  object        
 8   Vehicle           106930 non-null  int64         
 9   Code_Description  106930 non-null  object        
 10  Code_Category     105472 non-null  object        
 11  Datetime          106930 non-null  datetime64[ns]
 12  Year              106930 non-null  int32         
 13  Month             106930 non-null  int32         
 14  Day_

In [ ]:
df.describe()

,Min Delay,Min Gap,Vehicle,Datetime,Year,Month,Day_Num,Hour,Weekday,Peak_Hour,Weekend
count,106930.000000,106930.000000,106930.000000,106930,106930.000000,106930.000000,106930.000000,106930.000000,106930.000000,106930.000000,106930.000000
mean,2.947592,4.228523,3327.152885,2024-06-05 18:00:37.272047360,2023.960310,6.132330,15.794847,13.000888,2.852062,0.392593,0.243458
min,0.000000,0.000000,0.000000,2022-01-01 01:02:00,2022.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,2023-05-17 00:33:45,2023.000000,3.000000,8.000000,8.000000,1.000000,0.000000,0.000000
50%,0.000000,0.000000,5141.000000,2024-06-29 06:05:00,2024.000000,6.000000,16.000000,14.000000,3.000000,0.000000,0.000000
75%,4.000000,8.000000,5616.000000,2025-07-16 11:17:45,2025.000000,9.000000,23.000000,18.000000,4.000000,1.000000,0.000000
max,900.000000,906.000000,9546.000000,2026-06-30 23:55:00,2026.000000,12.000000,31.000000,23.000000,6.000000,1.000000,1.000000
std,11.134875,11.047004,2726.253945,NaN,1.299452,3.444323,8.785973,6.571480,1.940412,0.488330,0.429171


In [ ]:
#Cleaned Dataset written into CSV file for further analysis
df.to_csv(EXPORT_DIR /"TTC_Subway_Delay_Cleaned_2022_June2026.csv")